# DLGenAI Project 26T3 - Milestone 5
## Post-Processing, Test-Time Augmentation, Ensembling & Production Inference

### Overview & Theoretical Foundations
Milestone 5 is where a trained segmentation model is turned into a final, competition-ready submission. Building on the architectures and optimization dynamics trained in earlier milestones, this milestone covers:

1. **Probability Maps to Binary Masks**:
   - Converting continuous sigmoid probability maps $\mathbf{P} \in [0, 1]^{C \times H \times W}$ into discrete binary decisions $\hat{\mathbf{X}} \in \{0, 1\}^{C \times H \times W}$.
   - The strict $(p > T)$ vs. non-strict $(p \ge T)$ boundary convention and logit-probability equivalence ($z = \ln(p / (1-p))$).
   - Global vs. per-class threshold calibration: balancing the trade-off between Foreground Dice ($70\%$) and False Positive / Empty Sheet Accuracy ($30\%$).

2. **Connected Component Area Filtering**:
   - Morphological cleanup and 8-connectivity labeling via `cv2.connectedComponentsWithStats`.
   - Filtering out small false-positive speckles to turn noisy defect predictions into completely empty sheets ($1.0$ Dice on clean sheets).
   - Tuning minimum area thresholds per defect class to preserve genuinely fine defects (e.g. pitting/inclusions vs. broad patches).

3. **Test-Time Augmentation (TTA)**:
   - Generating flipped tensor variations (horizontal `dims=[3]`, vertical `dims=[2]`, and combined `dims=[2, 3]`).
   - Applying exact spatial inverse-transforms before averaging in probability space to reduce prediction variance.

4. **Ensembling & Multi-Model Blending**:
   - Soft probability averaging across diverse checkpoints and backbones.
   - Probability-space vs. logit-space fusion and auxiliary classifier probability gating.

5. **Production Inference Under Kaggle Constraints**:
   - Strict GPU RAM and CPU memory management (running-sum accumulation, float16/uint8 footprint, garbage collection).
   - Strict order of operations: **TTA $\to$ Ensemble $\to$ Threshold $\to$ Area Filter $\to$ Run-Length Encoding (RLE)**.

In [1]:
# Common Environment & Path Setup
import os
import sys
import time
import gc
import ctypes
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Apply glibc memory optimizations if running on Linux / Kaggle
try:
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-100), ctypes.c_int(0))
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-101), ctypes.c_int(1))
except Exception:
    pass

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

class CFG:
    SEED = 42
    CANVAS_HEIGHT = 128
    CANVAS_WIDTH = 800
    CANVAS_SIZE = 128 * 800  # 102,400
    DEFECT_CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
    
    # Calibrated decision thresholds
    CLASS_THRESHOLDS = {0: 0.32, 1: 0.35, 2: 0.30, 3: 0.30}
    # Calibrated minimum component areas (in pixels)
    MIN_AREAS = {0: 60, 1: 100, 2: 350, 3: 250}

# Path Resolution
def resolve_paths():
    candidates = [
        ('/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/competitions/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/train.csv', '/kaggle/input/industrial-surface-defect-detection/train'),
        ('data/train.csv', 'data/train'),
        ('../data/train.csv', '../data/train'),
    ]
    for csv_p, img_p in candidates:
        if os.path.exists(csv_p):
            return csv_p, img_p
    return 'data/train.csv', 'data/train'

DATA_CSV, IMAGE_DIR = resolve_paths()
print(f"[Setup] DATA_CSV: {DATA_CSV}")
print(f"[Setup] IMAGE_DIR: {IMAGE_DIR}")


[Setup] DATA_CSV: data/train.csv
[Setup] IMAGE_DIR: data/train


In [2]:
# RLE Encoding & Metric Helper Functions
def rle_to_mask(rle_string, height=128, width=800):
    """Decodes RLE mask on native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)."""
    if pd.isna(rle_string) or not str(rle_string).strip():
        return np.zeros((height, width), dtype=np.uint8)
    canvas_size = height * width
    runs = np.asarray([int(x) for x in str(rle_string).strip().split()], dtype=np.int32)
    starts = runs[0::2] - 1
    lengths = runs[1::2]
    mask = np.zeros(canvas_size, dtype=np.uint8)
    for start, length in zip(starts, lengths):
        if start >= canvas_size:
            continue
        end = min(start + length, canvas_size)
        mask[start:end] = 1
    return mask.reshape((height, width), order='F')

def mask_to_rle(mask):
    """Encodes binary mask (height, width) into competition-standard Fortran-ordered 1-based RLE string."""
    pixels = mask.flatten(order='F')
    if pixels.sum() == 0:
        return ''
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

def compute_dice_score(y_pred, y_true):
    """Computes Dice coefficient between binary predictions and ground truth."""
    intersection = np.sum(y_pred * y_true)
    total = np.sum(y_pred) + np.sum(y_true)
    if total == 0:
        return 1.0
    return 2.0 * float(intersection) / float(total)

print("[Setup] Utility functions defined successfully.")


[Setup] Utility functions defined successfully.


---
## Question 1: Logits, Probability Thresholds & the Strict-Inequality Trap

### Problem Statement
A model's final layer outputs raw logits (no sigmoid applied yet) for 6 pixels of one image-class pair:

| Pixel | 1 | 2 | 3 | 4 | 5 | 6 |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Logit ($z$)** | -2.0 | -0.9 | -0.5 | 0.0 | 0.8 | 2.2 |
| **Ground Truth ($Y$)** | 0 | 0 | 1 | 1 | 1 | 0 |

### Questions:
1. **Which logit value corresponds exactly to a probability threshold of $T = 0.3$ (4 decimal places)?**
   $$\sigma(z) = \frac{1}{1 + e^{-z}} = T \implies z = \ln\left(\frac{T}{1 - T}\right) = \ln\left(\frac{0.3}{0.7}\right) = \ln\left(\frac{3}{7}\right) \approx -0.8473$$

2. **At $T = 0.5$, how many pixels are predicted positive using strict ($p > T$), and how many using ($p \ge T$)?**
   - At $T = 0.5$, the corresponding logit threshold is $z = \ln(0.5 / 0.5) = \ln(1) = 0.0$.
   - For pixel 4 ($z = 0.0$), $p = \sigma(0.0) = 0.5000$.
   - Under strict inequality ($p > 0.5$ / $z > 0.0$): pixels 5 and 6 ($z = 0.8, 2.2$) are positive $\implies n_{\text{strict}} = 2$.
   - Under non-strict inequality ($p \ge 0.5$ / $z \ge 0.0$): pixels 4, 5, and 6 ($z = 0.0, 0.8, 2.2$) are positive $\implies n_{\ge} = 3$.

3. **Dice score at $T = 0.3$ (strict), and Dice score at $T = 0.5$ (strict) (4 decimal places)?**
   - Ground Truth: $Y = [0, 0, 1, 1, 1, 0]$ with $|Y| = 3$ positive pixels (indices 3, 4, 5).
   - **At $T = 0.3$ (strict)**: $z > -0.8473$.
     - Positive pixels: 3, 4, 5, 6 $\implies |\hat{X}| = 4$.
     - Intersection: $|\hat{X} \cap Y| = 3$ (pixels 3, 4, 5).
     - $\text{Dice}_{0.3} = \frac{2 \times 3}{4 + 3} = \frac{6}{7} \approx 0.8571$.
   - **At $T = 0.5$ (strict)**: $z > 0.0$.
     - Positive pixels: 5, 6 $\implies |\hat{X}| = 2$.
     - Intersection: $|\hat{X} \cap Y| = 1$ (pixel 5).
     - $\text{Dice}_{0.5} = \frac{2 \times 1}{2 + 3} = \frac{2}{5} = 0.4000$.

**Answer format:** `logit_T0.3, n_strict, n_geq, Dice@0.3, Dice@0.5`
**Final Result:** `-0.8473, 2, 3, 0.8571, 0.4000`

In [3]:
# ==============================================================================
# Question 1: Logits, Probability Thresholds & the Strict-Inequality Trap
# ==============================================================================
pixels = np.array([1, 2, 3, 4, 5, 6])
logits = np.array([-2.0, -0.9, -0.5, 0.0, 0.8, 2.2])
truth = np.array([0, 0, 1, 1, 1, 0])

# 1. Logit value corresponding to probability threshold T = 0.3
# Using logit(p) = ln(p / (1 - p))
T_target = 0.3
logit_T03 = float(np.log(T_target / (1.0 - T_target)))

# Compute sigmoid probabilities for each pixel
probs = 1.0 / (1.0 + np.exp(-logits))

print("--- Pixel Activations ---")
for px, z, p, y in zip(pixels, logits, probs, truth):
    print(f"Pixel {px}: logit = {z:5.2f} -> prob = {p:.6f} | Ground Truth = {y}")

# 2. Strict (p > 0.5) vs Non-Strict (p >= 0.5)
pred_strict_05 = (probs > 0.5).astype(int)
pred_geq_05 = (probs >= 0.5).astype(int)
n_strict = int(pred_strict_05.sum())
n_geq = int(pred_geq_05.sum())

# 3. Dice score at T = 0.3 (strict) and T = 0.5 (strict)
pred_strict_03 = (probs > 0.3).astype(int)

def dice(pred, y):
    intersection = (pred * y).sum()
    denom = pred.sum() + y.sum()
    return 1.0 if denom == 0 else 2.0 * intersection / denom

dice_03 = dice(pred_strict_03, truth)
dice_05 = dice(pred_strict_05, truth)

print("\n--- Question 1 Sub-Answers ---")
print(f"1. Logit for T=0.3: {logit_T03:.4f}")
print(f"2. At T=0.5 -> n_strict (p > 0.5): {n_strict}, n_geq (p >= 0.5): {n_geq}")
print(f"3. Dice@0.3 (strict): {dice_03:.4f}, Dice@0.5 (strict): {dice_05:.4f}")

# Final Formatted Answer
q1_answer = f"{logit_T03:.4f}, {n_strict}, {n_geq}, {dice_03:.4f}, {dice_05:.4f}"
print("\n" + "="*40)
print(f"Question 1 Final Answer: {q1_answer}")
print("="*40)


--- Pixel Activations ---
Pixel 1: logit = -2.00 -> prob = 0.119203 | Ground Truth = 0
Pixel 2: logit = -0.90 -> prob = 0.289050 | Ground Truth = 0
Pixel 3: logit = -0.50 -> prob = 0.377541 | Ground Truth = 1
Pixel 4: logit =  0.00 -> prob = 0.500000 | Ground Truth = 1
Pixel 5: logit =  0.80 -> prob = 0.689974 | Ground Truth = 1
Pixel 6: logit =  2.20 -> prob = 0.900250 | Ground Truth = 0

--- Question 1 Sub-Answers ---
1. Logit for T=0.3: -0.8473
2. At T=0.5 -> n_strict (p > 0.5): 2, n_geq (p >= 0.5): 3
3. Dice@0.3 (strict): 0.8571, Dice@0.5 (strict): 0.4000

Question 1 Final Answer: -0.8473, 2, 3, 0.8571, 0.4000


---
## Question 2: Per-Class Threshold / Min-Area Selection from a Sweep Table

### Problem Statement
For `defect_b`, a student sweeps three post-processing configs on the holdout and records:

| Config | $T$ | $N$ (min area) | FgDice | FalsePositiveScore | EmptyAccuracy |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **X** | 0.5 | 0 | 0.62 | 0.9950 | 0.40 |
| **Y** | 0.7 | 100 | 0.58 | 0.9990 | 0.85 |
| **Z** | 0.9 | 300 | 0.45 | 0.9998 | 0.97 |

### Official Score Formula:
$$\text{Score} = 0.70 \times \text{FgDice} + 0.20 \times \text{FalsePositiveScore} + 0.10 \times \text{EmptyAccuracy}$$

### Calculations:
1. **Config X**:
   $$\text{Score}_X = 0.70 \times 0.62 + 0.20 \times 0.9950 + 0.10 \times 0.40 = 0.4340 + 0.1990 + 0.0400 = 0.6730$$
2. **Config Y**:
   $$\text{Score}_Y = 0.70 \times 0.58 + 0.20 \times 0.9990 + 0.10 \times 0.85 = 0.4060 + 0.1998 + 0.0850 = 0.6908$$
3. **Config Z**:
   $$\text{Score}_Z = 0.70 \times 0.45 + 0.20 \times 0.9998 + 0.10 \times 0.97 = 0.3150 + 0.19996 + 0.0970 = 0.61196 \approx 0.6120$$

### Analysis:
- **Best Config**: Config Y achieves the highest overall official score ($0.6908$).
- **Why it beats Config X (highest FgDice = 0.62)**:
  - Config X suffers from small false positive speckles on empty sheets, crashing `EmptyAccuracy` to $0.40$.
  - Config Y introduces threshold tuning ($T = 0.7$) and minimum area filtering ($N = 100$), which dramatically improves `EmptyAccuracy` from $0.40 \to 0.85$ ($+0.0450$ in weighted score) and `FalsePositiveScore` from $0.9950 \to 0.9990$ ($+0.0008$).
  - This $+0.0458$ gain on empty sheets easily overcomes the small $-0.0280$ penalty from FgDice dropping from $0.62 \to 0.58$.
  - Therefore, the two components that explain why Y beats X are **FalsePositiveScore** and **EmptyAccuracy**.

**Answer format:** `Score_X, Score_Y, Score_Z, best config, components`
**Final Result:** `0.6730, 0.6908, 0.6120, Y, FalsePositiveScore, EmptyAccuracy`

In [4]:
# ==============================================================================
# Question 2: Per-Class Threshold / Min-Area Selection from a Sweep Table
# ==============================================================================
configs = {
    'X': {'T': 0.5, 'N': 0,   'FgDice': 0.62, 'FP': 0.9950, 'EmptyAcc': 0.40},
    'Y': {'T': 0.7, 'N': 100, 'FgDice': 0.58, 'FP': 0.9990, 'EmptyAcc': 0.85},
    'Z': {'T': 0.9, 'N': 300, 'FgDice': 0.45, 'FP': 0.9998, 'EmptyAcc': 0.97},
}

def compute_official_score(fg, fp, emp):
    return 0.70 * fg + 0.20 * fp + 0.10 * emp

scores = {}
print('--- Official Score Calculations ---')
for name, cfg in configs.items():
    s = compute_official_score(cfg['FgDice'], cfg['FP'], cfg['EmptyAcc'])
    scores[name] = s
    print(f"Config {name} (T={cfg['T']}, N={cfg['N']}): Score = 0.70*{cfg['FgDice']} + 0.20*{cfg['FP']} + 0.10*{cfg['EmptyAcc']} = {s:.4f}")

score_X = scores['X']
score_Y = scores['Y']
score_Z = scores['Z']

# Determine best config
best_config = max(scores, key=scores.get)

# Compare Config Y vs Config X (highest FgDice)
delta_fg = 0.70 * (configs['Y']['FgDice'] - configs['X']['FgDice'])
delta_fp = 0.20 * (configs['Y']['FP'] - configs['X']['FP'])
delta_emp = 0.10 * (configs['Y']['EmptyAcc'] - configs['X']['EmptyAcc'])

print('\n--- Comparison between Best Config (Y) and Highest FgDice Config (X) ---')
print(f'FgDice contribution change:         {delta_fg:+.4f}')
print(f'FalsePositiveScore contribution:    {delta_fp:+.4f}')
print(f'EmptyAccuracy contribution:         {delta_emp:+.4f}')
print(f"Net Score change (Y - X):           {scores['Y'] - scores['X']:+.4f}")

components_str = 'FalsePositiveScore, EmptyAccuracy'

# Final Formatted Answer
q2_answer = f'{score_X:.4f}, {score_Y:.4f}, {score_Z:.4f}, {best_config}, {components_str}'
print('\n' + '='*50)
print(f'Question 2 Final Answer: {q2_answer}')
print('='*50)


--- Official Score Calculations ---
Config X (T=0.5, N=0): Score = 0.70*0.62 + 0.20*0.995 + 0.10*0.4 = 0.6730
Config Y (T=0.7, N=100): Score = 0.70*0.58 + 0.20*0.999 + 0.10*0.85 = 0.6908
Config Z (T=0.9, N=300): Score = 0.70*0.45 + 0.20*0.9998 + 0.10*0.97 = 0.6120

--- Comparison between Best Config (Y) and Highest FgDice Config (X) ---
FgDice contribution change:         -0.0280
FalsePositiveScore contribution:    +0.0008
EmptyAccuracy contribution:         +0.0450
Net Score change (Y - X):           +0.0178

Question 2 Final Answer: 0.6730, 0.6908, 0.6120, Y, FalsePositiveScore, EmptyAccuracy


---
## Question 3: Connected Components, Connectivity & Area Filtering

### Problem Statement
A predicted binary mask on a $6 \times 8$ canvas ($1 =$ predicted defect):
```text
1 1 0 0 0 0 0 1
1 1 0 0 0 0 0 0
0 0 1 0 0 0 0 0
0 0 0 1 0 0 0 0
0 0 0 0 0 1 1 0
1 0 0 0 0 1 1 1
```

### Questions:
1. **How many connected components does this mask have under 8-connectivity? How many under 4-connectivity?**
   - **8-connectivity ($n_{\text{8conn}} = 4$)**:
     - Component 1: Top-left cluster + diagonal chain `{(0,0), (0,1), (1,0), (1,1), (2,2), (3,3)}` $\implies \text{area} = 6$.
     - Component 2: Top-right isolated pixel `{(0,7)}` $\implies \text{area} = 1$.
     - Component 3: Bottom-left isolated pixel `{(5,0)}` $\implies \text{area} = 1$.
     - Component 4: Bottom-right block `{(4,5), (4,6), (5,5), (5,6), (5,7)}` $\implies \text{area} = 5$.
     - Total components under 8-connectivity = **4**.
   - **4-connectivity ($n_{\text{4conn}} = 6$)**:
     - Diagonal pixels are not adjacent in 4-connectivity, splitting the top-left structure:
       - 2x2 square `{(0,0), (0,1), (1,0), (1,1)}` $\implies \text{area} = 4$.
       - Pixel `{(2,2)}` $\implies \text{area} = 1$.
       - Pixel `{(3,3)}` $\implies \text{area} = 1$.
     - Top-right pixel `{(0,7)}` $\implies \text{area} = 1$.
     - Bottom-left pixel `{(5,0)}` $\implies \text{area} = 1$.
     - Bottom-right block `{(4,5), (4,6), (5,5), (5,6), (5,7)}` $\implies \text{area} = 5$.
     - Total components under 4-connectivity = **6**.

2. **Using 8-connectivity, if you discard every component with area $< 5$, how many foreground pixels remain?**
   - Components with area $< 5$: Component 2 (area 1) and Component 3 (area 1) are discarded.
   - Components with area $\ge 5$: Component 1 (area 6) and Component 4 (area 5) are kept.
   - Remaining pixels = $6 + 5 = \mathbf{11}$.

3. **Using 8-connectivity, if you discard every component with area $< 6$, how many foreground pixels remain?**
   - Components with area $< 6$: Component 2 (area 1), Component 3 (area 1), and Component 4 (area 5) are discarded.
   - Component with area $\ge 6$: Component 1 (area 6) is kept.
   - Remaining pixels = $\mathbf{6}$.

**Answer format:** `n_8conn, n_4conn, pixels_N5, pixels_N6`
**Final Result:** `4, 6, 11, 6`

In [5]:
# ==============================================================================
# Question 3: Connected Components, Connectivity & Area Filtering
# ==============================================================================
mask_q3 = np.array([
    [1, 1, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0],
    [1, 0, 0, 0, 0, 1, 1, 1],
], dtype=np.uint8)

# 1. Connected components under 8-connectivity
n_labels_8, labels_8, stats_8, _ = cv2.connectedComponentsWithStats(mask_q3, connectivity=8)
n_8conn = n_labels_8 - 1
areas_8 = stats_8[1:, cv2.CC_STAT_AREA]

# Connected components under 4-connectivity
n_labels_4, labels_4, stats_4, _ = cv2.connectedComponentsWithStats(mask_q3, connectivity=4)
n_4conn = n_labels_4 - 1
areas_4 = stats_4[1:, cv2.CC_STAT_AREA]

# 2. Discard area < 5 under 8-connectivity
pixels_N5 = int(sum(a for a in areas_8 if a >= 5))

# 3. Discard area < 6 under 8-connectivity
pixels_N6 = int(sum(a for a in areas_8 if a >= 6))

print(f"8-connectivity components: {n_8conn} (areas: {sorted(areas_8.tolist(), reverse=True)})")
print(f"4-connectivity components: {n_4conn} (areas: {sorted(areas_4.tolist(), reverse=True)})")
print(f"Remaining pixels with area >= 5: {pixels_N5}")
print(f"Remaining pixels with area >= 6: {pixels_N6}")

q3_answer = f"{n_8conn}, {n_4conn}, {pixels_N5}, {pixels_N6}"
print("\n" + "="*40)
print(f"Question 3 Final Answer: {q3_answer}")
print("="*40)


8-connectivity components: 4 (areas: [6, 5, 1, 1])
4-connectivity components: 6 (areas: [5, 4, 1, 1, 1, 1])
Remaining pixels with area >= 5: 11
Remaining pixels with area >= 6: 6

Question 3 Final Answer: 4, 6, 11, 6


---
## Question 4: Does Post-Processing Actually Improve the Official Score?

### Problem Statement
A validation set has 100 image-class pairs: 20 positive and 80 empty ($|Y| = 0$). The canvas is $128 \times 800 = 102,400$ pixels. Before post-processing:
- **Positive pairs**: 12 pairs have Dice 0.70 and 8 pairs have Dice 0.45.
- **Empty pairs**: 50 are predicted completely empty. 20 contain a single 20-pixel speckle each. 10 contain a single 2,000-pixel blob each.

A minimum-area filter with $N = 50$ (discard components with area $< 50$) is applied:
- Removes all 20 speckles (area $20 < 50$), converting them to completely empty masks.
- Does not touch the 2,000-pixel blobs (area $2000 \ge 50$).
- Deletes the only predicted component (30 px) of one of the positive pairs that had Dice 0.45, reducing that pair's Dice to 0.

### Official Metric Formula:
$$\text{Score} = 0.70 \times \text{ForegroundDice} + 0.20 \times \text{FalsePositiveScore} + 0.10 \times \text{EmptyAccuracy}$$
$$\text{FalsePositiveScore} = 1 - \text{mean}\left(\frac{|X|}{102,400}\right) \quad \text{over empty pairs}$$

### Step-by-Step Derivations:
1. **Before Filtering**:
   - $\text{FgDice}_{\text{before}} = \frac{12 \times 0.70 + 8 \times 0.45}{20} = \frac{8.40 + 3.60}{20} = 0.6000$
   - Empty pairs false positive pixels: $50 \times 0 + 20 \times 20 + 10 \times 2000 = 20,400$ px.
   - $\text{Mean } |X| = 20,400 / 80 = 255$ px $\implies \text{FP}_{\text{before}} = 1 - \frac{255}{102,400} \approx 0.9975098 \approx 0.9975$
   - $\text{EmptyAcc}_{\text{before}} = 50 / 80 = 0.6250$
   - $\text{Score}_{\text{before}} = 0.70(0.6000) + 0.20(0.9975098) + 0.10(0.6250) = 0.4200 + 0.199502 + 0.0625 = 0.682002 \approx 0.6820$

2. **After Filtering ($N = 50$)**:
   - $\text{FgDice}_{\text{after}} = \frac{12 \times 0.70 + 7 \times 0.45 + 1 \times 0.0}{20} = \frac{8.40 + 3.15 + 0.0}{20} = \frac{11.55}{20} = 0.5775$
   - 20 speckles eliminated $\implies 50 + 20 = 70$ empty pairs now completely empty ($|X| = 0$).
   - 10 empty pairs retain the 2,000 px blob $\implies$ total FP pixels = $10 \times 2000 = 20,000$ px.
   - $\text{Mean } |X| = 20,000 / 80 = 250$ px $\implies \text{FP}_{\text{after}} = 1 - \frac{250}{102,400} \approx 0.9975586 \approx 0.9976$
   - $\text{EmptyAcc}_{\text{after}} = 70 / 80 = 0.8750$
   - $\text{Score}_{\text{after}} = 0.70(0.5775) + 0.20(0.9975586) + 0.10(0.8750) = 0.40425 + 0.199512 + 0.0875 = 0.691262 \approx 0.6913$

**Answer format:** `FgDice_after, FP_after, EmptyAcc_after, Score_before, Score_after`
**Final Result:** `0.5775, 0.9976, 0.8750, 0.6820, 0.6913`

In [6]:
# ==============================================================================
# Question 4: Does Post-Processing Actually Improve the Official Score?
# ==============================================================================
canvas_size = 102400

# Before filtering
fg_dice_before = (12 * 0.70 + 8 * 0.45) / 20.0
empty_preds_before = [0]*50 + [20]*20 + [2000]*10
fp_score_before = 1.0 - (sum(empty_preds_before) / len(empty_preds_before)) / canvas_size
empty_acc_before = 50 / 80.0
score_before = 0.70 * fg_dice_before + 0.20 * fp_score_before + 0.10 * empty_acc_before

# After filtering (N = 50)
fg_dice_after = (12 * 0.70 + 7 * 0.45 + 1 * 0.0) / 20.0
empty_preds_after = [0]*70 + [2000]*10
fp_score_after = 1.0 - (sum(empty_preds_after) / len(empty_preds_after)) / canvas_size
empty_acc_after = 70 / 80.0
score_after = 0.70 * fg_dice_after + 0.20 * fp_score_after + 0.10 * empty_acc_after

print("--- Metric Values Before Filtering ---")
print(f"FgDice: {fg_dice_before:.4f} | FP Score: {fp_score_before:.4f} | Empty Acc: {empty_acc_before:.4f} | Official Score: {score_before:.4f}")

print("\n--- Metric Values After Filtering (N = 50) ---")
print(f"FgDice: {fg_dice_after:.4f} | FP Score: {fp_score_after:.4f} | Empty Acc: {empty_acc_after:.4f} | Official Score: {score_after:.4f}")
print(f"Score Improvement: {score_after - score_before:+.4f}")

q4_answer = f"{fg_dice_after:.4f}, {fp_score_after:.4f}, {empty_acc_after:.4f}, {score_before:.4f}, {score_after:.4f}"
print("\n" + "="*40)
print(f"Question 4 Final Answer: {q4_answer}")
print("="*40)


--- Metric Values Before Filtering ---
FgDice: 0.6000 | FP Score: 0.9975 | Empty Acc: 0.6250 | Official Score: 0.6820

--- Metric Values After Filtering (N = 50) ---
FgDice: 0.5775 | FP Score: 0.9976 | Empty Acc: 0.8750 | Official Score: 0.6913
Score Improvement: +0.0093

Question 4 Final Answer: 0.5775, 0.9976, 0.8750, 0.6820, 0.6913


---
## Question 5: Recall Ceiling of Min-Area Filtering at the Pair Level

### Problem Statement
Decode every non-empty mask in `train.csv` and run 8-connected component analysis on each pair.
A positive pair is *wiped out* by a filter with threshold $N$ if every one of its components has area $< N$ (i.e. $\max(\text{areas}) < N$), meaning it becomes empty after filtering regardless of model quality.

### Questions:
1. **With $N = 100$, how many positive image-class pairs (across all 4 classes) are wiped out?**
   - In the full dataset `train.csv`, there are 7,201 positive image-class pairs.
   - Running 8-connectivity component analysis (`cv2.connectedComponentsWithStats`) on all 7,201 masks reveals that **48** positive pairs have all components with area $< 100$.

2. **Which defect class loses the most pairs?**
   - Breakdown of wiped-out pairs by class:
     - `defect_a`: 22 / 1,081 pairs (2.04%)
     - `defect_b`: 1 / 313 pairs (0.32%)
     - `defect_c`: **24** / 4,100 pairs (0.59%)
     - `defect_d`: 1 / 1,707 pairs (0.06%)
   - `defect_c` loses the highest absolute number of positive pairs (24 pairs).

**Answer format:** `Integer, defect name`
**Final Result:** `48, defect_c`

In [7]:
# ==============================================================================
# Question 5: Recall Ceiling of Min-Area Filtering at the Pair Level
# ==============================================================================
df_train = pd.read_csv(DATA_CSV)
pos_pairs = df_train[df_train['mask_rle'].notna() & (df_train['mask_rle'].astype(str).str.strip() != '')].copy()

N_thresh = 100
wiped_out_by_class = {'defect_a': 0, 'defect_b': 0, 'defect_c': 0, 'defect_d': 0}
total_by_class = {'defect_a': 0, 'defect_b': 0, 'defect_c': 0, 'defect_d': 0}

for idx, row in pos_pairs.iterrows():
    cls = row['defect_type']
    total_by_class[cls] += 1
    rle = row['mask_rle']
    mask = rle_to_mask(rle, CFG.CANVAS_HEIGHT, CFG.CANVAS_WIDTH)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    comp_areas = stats[1:, cv2.CC_STAT_AREA]
    if len(comp_areas) == 0 or max(comp_areas) < N_thresh:
        wiped_out_by_class[cls] += 1

total_wiped_out = sum(wiped_out_by_class.values())
most_lost_class = max(wiped_out_by_class, key=wiped_out_by_class.get)

print(f"--- Wiped Out Positive Pairs with N = {N_thresh} (8-connectivity) ---")
for cls, cnt in wiped_out_by_class.items():
    tot = total_by_class[cls]
    print(f"{cls:10s}: {cnt:3d} / {tot:4d} pairs wiped out ({cnt/tot*100:.2f}%)")

print(f"\nTotal wiped out pairs across all 4 classes: {total_wiped_out}")
print(f"Defect class losing the most pairs: {most_lost_class} ({wiped_out_by_class[most_lost_class]} pairs)")

q5_answer = f"{total_wiped_out}, {most_lost_class}"
print("\n" + "="*40)
print(f"Question 5 Final Answer: {q5_answer}")
print("="*40)


--- Wiped Out Positive Pairs with N = 100 (8-connectivity) ---
defect_a  :  22 / 1081 pairs wiped out (2.04%)
defect_b  :   1 /  313 pairs wiped out (0.32%)
defect_c  :  24 / 4100 pairs wiped out (0.59%)
defect_d  :   1 / 1707 pairs wiped out (0.06%)

Total wiped out pairs across all 4 classes: 48
Defect class losing the most pairs: defect_c (24 pairs)

Question 5 Final Answer: 48, defect_c


---
## Question 6: Test-Time Augmentation: Flip Axes and the "Forgot to Un-flip" Bug

### Problem Statement
Input tensors have shape $(B, C, H, W) = (B, 1, 128, 800)$. For one $2 \times 2$ patch of a single class, the model produced:
- **Original pass**: `[[0.9, 0.6], [0.4, 0.2]]`
- **Horizontal-flip pass** (raw output, in the flipped frame): `[[0.5, 0.8], [0.3, 0.4]]`
- **Vertical-flip pass** (raw output, in the flipped frame): `[[0.1, 0.5], [0.7, 0.9]]`

### Questions & Derivations:
1. **Which dims list does `torch.flip` need for a horizontal flip and for a vertical flip on this tensor layout?**
   - In a 4D tensor with shape $(B, C, H, W)$:
     - Index 0: Batch ($B$)
     - Index 1: Channel ($C$)
     - Index 2: Height / Vertical ($H$)
     - Index 3: Width / Horizontal ($W$)
   - A horizontal flip mirrors along the width axis $\implies$ `hflip_dim = 3` (or `[3]`, or 1 for 2D spatial).
   - A vertical flip mirrors along the height axis $\implies$ `vflip_dim = 2` (or `[2]`, or 0 for 2D spatial).

2. **After correctly inverting each flip and averaging the three passes, what is the averaged probability at the top-right pixel (4 d.p.)?**
   - Original: `[[0.9, 0.6], [0.4, 0.2]]` $\implies$ top-right = 0.6
   - Horizontal un-flip (mirror columns left $\leftrightarrow$ right):
     - Row 0: `[0.5, 0.8]` becomes `[0.8, 0.5]` $\implies$ top-right = 0.5
     - Row 1: `[0.3, 0.4]` becomes `[0.4, 0.3]`
   - Vertical un-flip (mirror rows top $\leftrightarrow$ bottom):
     - Row 0 was `[0.1, 0.5]`, Row 1 was `[0.7, 0.9]`
     - After vertical flip: Row 0 becomes `[0.7, 0.9]` $\implies$ top-right = 0.9
   - Correct Averaged Top-Right: $\frac{0.6 + 0.5 + 0.9}{3} = \frac{2.0}{3} \approx \mathbf{0.6667}$.

3. **At $T = 0.5$ (strict $>$), how many pixels in the $2 \times 2$ patch are positive with the correct pipeline?**
   - Full Correct Averaged Patch:
     - Top-left: $(0.9 + 0.8 + 0.7)/3 = 2.4/3 = 0.8000 > 0.5$ (Positive)
     - Top-right: $(0.6 + 0.5 + 0.9)/3 = 2.0/3 = 0.6667 > 0.5$ (Positive)
     - Bottom-left: $(0.4 + 0.4 + 0.1)/3 = 0.9/3 = 0.3000 \le 0.5$ (Negative)
     - Bottom-right: $(0.2 + 0.3 + 0.5)/3 = 1.0/3 = 0.3333 \le 0.5$ (Negative)
   - Number of positive pixels = **2** (`n_correct = 2`).

4. **How many are positive if the student forgets to inverse-flip the two augmented outputs before averaging?**
   - Buggy Averaged Patch (averaging raw outputs directly):
     - Top-left: $(0.9 + 0.5 + 0.1)/3 = 1.5/3 = 0.5000$. Under strict $(p > 0.5)$, $0.5000 > 0.5$ is **False**!
     - Top-right: $(0.6 + 0.8 + 0.5)/3 = 1.9/3 = 0.6333 > 0.5$ (Positive)
     - Bottom-left: $(0.4 + 0.3 + 0.7)/3 = 1.4/3 = 0.4667 \le 0.5$ (Negative)
     - Bottom-right: $(0.2 + 0.4 + 0.9)/3 = 1.5/3 = 0.5000 \le 0.5$ (Negative)
   - Only the top-right pixel survives $\implies$ **1** (`n_buggy = 1`).

**Answer format:** `hflip_dim, vflip_dim, top-right_value, n_correct, n_buggy`
**Final Result:** `3, 2, 0.6667, 2, 1` *(or `1, 0, 0.6667, 2, 1` if spatial dims only)*

In [8]:
# ==============================================================================
# Question 6: Test-Time Augmentation: Flip Axes and the "Forgot to Un-flip" Bug
# ==============================================================================
# Tensor layout: (B, C, H, W) = (B, 1, 128, 800)
hflip_dim = 3  # width axis (dim 3)
vflip_dim = 2  # height axis (dim 2)

# 2x2 patch outputs
orig = np.array([[0.9, 0.6], [0.4, 0.2]])
h_raw = np.array([[0.5, 0.8], [0.3, 0.4]])
v_raw = np.array([[0.1, 0.5], [0.7, 0.9]])

# Correct pipeline: invert flips before averaging
h_inv = np.fliplr(h_raw)
v_inv = np.flipud(v_raw)
correct_avg = (orig + h_inv + v_inv) / 3.0

top_right_value = correct_avg[0, 1]
n_correct = int((correct_avg > 0.5).sum())

# Buggy pipeline: forgets to inverse-flip
buggy_avg = (orig + h_raw + v_raw) / 3.0
n_buggy = int((buggy_avg > 0.5).sum())

print("--- Correct TTA Averaging ---")
print(f"Correct Averaged Patch:\n{correct_avg}")
print(f"Top-right value: {top_right_value:.4f}")
print(f"Positive pixels (p > 0.5): {n_correct}")

print("\n--- Buggy TTA Averaging (Forgot Un-flip) ---")
print(f"Buggy Averaged Patch:\n{buggy_avg}")
print(f"Positive pixels (p > 0.5): {n_buggy}")

# Final Formatted Answer
q6_answer = f"{hflip_dim}, {vflip_dim}, {top_right_value:.4f}, {n_correct}, {n_buggy}"
print("\n" + "="*40)
print(f"Question 6 Final Answer: {q6_answer}")
print("="*40)


--- Correct TTA Averaging ---
Correct Averaged Patch:
[[0.8        0.66666667]
 [0.3        0.33333333]]
Top-right value: 0.6667
Positive pixels (p > 0.5): 2

--- Buggy TTA Averaging (Forgot Un-flip) ---
Buggy Averaged Patch:
[[0.5        0.63333333]
 [0.46666667 0.5       ]]
Positive pixels (p > 0.5): 1

Question 6 Final Answer: 3, 2, 0.6667, 2, 1


---
## Question 7: Ensembling: Soft Averaging, Hard Voting and Where to Average

### Problem Statement
Three models predict probabilities for 5 pixels of one image-class pair. Ground truth is positive at `p1, p3, p5` and negative at `p2, p4` ($Y = [1, 0, 1, 0, 1]$).

| Pixel | Model A | Model B | Model C | Ground Truth |
| :---: | :---: | :---: | :---: | :---: |
| **p1** | 0.90 | 0.40 | 0.45 | 1 |
| **p2** | 0.55 | 0.55 | 0.20 | 0 |
| **p3** | 0.80 | 0.70 | 0.90 | 1 |
| **p4** | 0.10 | 0.20 | 0.30 | 0 |
| **p5** | 0.60 | 0.45 | 0.55 | 1 |

Threshold $T = 0.5$ (strict $>$) everywhere.

### Step-by-Step Derivations:
1. **Dice of the soft ensemble (mean of probabilities, then threshold)**:
   - Mean probabilities:
     - p1: $(0.90 + 0.40 + 0.45)/3 = 0.5833 > 0.5 \implies 1$
     - p2: $(0.55 + 0.55 + 0.20)/3 = 0.4333 \le 0.5 \implies 0$
     - p3: $(0.80 + 0.70 + 0.90)/3 = 0.8000 > 0.5 \implies 1$
     - p4: $(0.10 + 0.20 + 0.30)/3 = 0.2000 \le 0.5 \implies 0$
     - p5: $(0.60 + 0.45 + 0.55)/3 = 0.5333 > 0.5 \implies 1$
   - $\hat{X}_{\text{soft}} = [1, 0, 1, 0, 1]$ (matches ground truth perfectly $\implies |\hat{X} \cap Y| = 3, |\hat{X}| = 3, |Y| = 3$).
   - $\text{Dice}_{\text{soft}} = \frac{2 \times 3}{3 + 3} = \mathbf{1.0000}$.

2. **Dice of the hard-vote ensemble (threshold each model at $T > 0.5$, then majority of 3)**:
   - Model individual predictions:
     - p1: A=1, B=0, C=0 $\implies$ sum = 1 $\implies$ majority = **0** (Missed defect!).
     - p2: A=1, B=1, C=0 $\implies$ sum = 2 $\implies$ majority = **1** (False alarm!).
     - p3: A=1, B=1, C=1 $\implies$ sum = 3 $\implies$ majority = **1** (Correct).
     - p4: A=0, B=0, C=0 $\implies$ sum = 0 $\implies$ majority = **0** (Correct).
     - p5: A=1, B=0, C=1 $\implies$ sum = 2 $\implies$ majority = **1** (Correct).
   - $\hat{X}_{\text{vote}} = [0, 1, 1, 0, 1]$ with $|\hat{X}| = 3$, $|\hat{X} \cap Y| = 2$ (pixels p3, p5).
   - $\text{Dice}_{\text{vote}} = \frac{2 \times 2}{3 + 3} = \frac{4}{6} \approx \mathbf{0.6667}$.

3. **Weighted ensemble with weights $(A, B, C) = (0.5, 0.3, 0.2)$ at p1**:
   $$p_1 = 0.5(0.90) + 0.3(0.40) + 0.2(0.45) = 0.45 + 0.12 + 0.09 = \mathbf{0.6600}$$

4. **Logits $+1$ and $-3$: probability space vs. logit space**:
   - **Probability space averaging**:
     $$\sigma(+1) = \frac{1}{1 + e^{-1}} \approx 0.731059, \quad \sigma(-3) = \frac{1}{1 + e^3} \approx 0.047426$$
     $$\text{prob\_avg} = \frac{0.731059 + 0.047426}{2} = \mathbf{0.3892}$$
   - **Logit space averaging**:
     $$z_{\text{avg}} = \frac{+1 + (-3)}{2} = -1.0 \implies \sigma(-1.0) = \frac{1}{1 + e^1} \approx \mathbf{0.2689}$$
   - At $T = 0.30$, `prob_avg` ($0.3892 > 0.30$) calls this pixel positive, while `logit_avg` ($0.2689 \le 0.30$) calls it negative.
   - Method calling positive: **`prob_avg`**.

**Answer format:** `Dice_soft, Dice_vote, weighted_p1, prob_avg, logit_avg, method`
**Final Result:** `1.0000, 0.6667, 0.6600, 0.3892, 0.2689, prob_avg`

In [9]:
# ==============================================================================
# Question 7: Ensembling: Soft Averaging, Hard Voting and Where to Average
# ==============================================================================
y_true = np.array([1, 0, 1, 0, 1])
probs = np.array([
    [0.90, 0.40, 0.45],  # p1
    [0.55, 0.55, 0.20],  # p2
    [0.80, 0.70, 0.90],  # p3
    [0.10, 0.20, 0.30],  # p4
    [0.60, 0.45, 0.55],  # p5
])

# 1. Soft ensemble
soft_mean = probs.mean(axis=1)
soft_pred = (soft_mean > 0.5).astype(int)
dice_soft = 2.0 * (soft_pred * y_true).sum() / (soft_pred.sum() + y_true.sum())

# 2. Hard-vote ensemble
model_preds = (probs > 0.5).astype(int)
vote_pred = (model_preds.sum(axis=1) >= 2).astype(int)
dice_vote = 2.0 * (vote_pred * y_true).sum() / (vote_pred.sum() + y_true.sum())

# 3. Weighted ensemble at p1
weights = np.array([0.5, 0.3, 0.2])
weighted_p1 = float((probs[0] * weights).sum())

# 4. Snapshot averaging (+1 and -3)
z1, z2 = 1.0, -3.0
p1 = 1.0 / (1.0 + np.exp(-z1))
p2 = 1.0 / (1.0 + np.exp(-z2))
prob_avg = float((p1 + p2) / 2.0)
logit_avg = float(1.0 / (1.0 + np.exp(-((z1 + z2) / 2.0))))
calling_method = 'prob_avg' if prob_avg > 0.30 else 'logit_avg'

print(f"Dice Soft: {dice_soft:.4f} | Dice Vote: {dice_vote:.4f}")
print(f"Weighted p1: {weighted_p1:.4f}")
print(f"Prob space avg: {prob_avg:.4f} | Logit space avg: {logit_avg:.4f} | Calling method @ 0.30: {calling_method}")

# Final Formatted Answer
q7_answer = f"{dice_soft:.4f}, {dice_vote:.4f}, {weighted_p1:.4f}, {prob_avg:.4f}, {logit_avg:.4f}, {calling_method}"
print("\n" + "="*40)
print(f"Question 7 Final Answer: {q7_answer}")
print("="*40)


Dice Soft: 1.0000 | Dice Vote: 0.6667
Weighted p1: 0.6600
Prob space avg: 0.3892 | Logit space avg: 0.2689 | Calling method @ 0.30: prob_avg

Question 7 Final Answer: 1.0000, 0.6667, 0.6600, 0.3892, 0.2689, prob_avg


---
## Question 8: Inference Budget: Memory and Time Across Two Ensemble Configurations

### Problem Statement
Hidden test set: $5,000$ images, each $128 \times 800$, with $4$ output classes.
One forward pass on one image takes $12\text{ ms} = 0.012\text{ s}$.
Decimal units: $1\text{ GB} = 10^9\text{ bytes}$.
- **Config A**: 3 models, 3-view TTA (original, hflip, vflip)
- **Config B**: 2 models, 2-view TTA (original, hflip)

### Detailed Derivations:
1. **Full test-set probability maps (5,000 images $\times$ 4 classes $\times$ 102,400 pixels = 2,048,000,000 elements)**:
   - **float32** (4 bytes/elem): $2,048,000,000 \times 4 = 8,192,000,000\text{ bytes} = \mathbf{8.192}\text{ GB} \approx \mathbf{8.2}\text{ GB}$
   - **float16** (2 bytes/elem): $2,048,000,000 \times 2 = 4,096,000,000\text{ bytes} = \mathbf{4.096}\text{ GB} \approx \mathbf{4.10}\text{ GB}$
   - **uint8** (1 byte/elem): $2,048,000,000 \times 1 = 2,048,000,000\text{ bytes} = \mathbf{2.048}\text{ GB} \approx \mathbf{2.05}\text{ GB}$

2. **Naively storing all (model, view) prediction sets in memory as float32**:
   - **Config A** (3 models $\times$ 3 views = 9 sets): $9 \times 8.192\text{ GB} = \mathbf{73.728}\text{ GB} \approx \mathbf{73.7}\text{ GB}$
   - **Config B** (2 models $\times$ 2 views = 4 sets): $4 \times 8.192\text{ GB} = \mathbf{32.768}\text{ GB} \approx \mathbf{32.8}\text{ GB}$
   - **Fits in 16 GB RAM budget?**: Neither fits ($73.7\text{ GB} > 16\text{ GB}$ and $32.8\text{ GB} > 16\text{ GB}$) $\implies$ **`neither`**.
   - **Single running float32 accumulator**: Keeps only 1 set in memory $\implies \mathbf{8.192}\text{ GB} \approx \mathbf{8.2}\text{ GB}$.

3. **Forward passes and pure model time (12 ms per pass)**:
   - **Config A**: $5,000 \times 3 \times 3 = \mathbf{45,000}$ passes $\implies 45,000 \times 0.012\text{ s} = 540\text{ s} = \mathbf{9.0}\text{ minutes}$.
   - **Config B**: $5,000 \times 2 \times 2 = \mathbf{20,000}$ passes $\implies 20,000 \times 0.012\text{ s} = 240\text{ s} = \mathbf{4.0}\text{ minutes}$.

4. **Maximum whole models under 30-min budget (8 min I/O $\implies$ 22 min = 1,320 s available)**:
   - **With 3-view TTA**: $5,000 \times 3 = 15,000$ passes/model $\implies 15,000 \times 0.012\text{ s} = 180\text{ s} = 3.0\text{ min/model}$.
     $$\lfloor 22 / 3.0 \rfloor = \lfloor 7.333 \rfloor = \mathbf{7}\text{ whole models}$$
   - **With 2-view TTA**: $5,000 \times 2 = 10,000$ passes/model $\implies 10,000 \times 0.012\text{ s} = 120\text{ s} = 2.0\text{ min/model}$.
     $$\lfloor 22 / 2.0 \rfloor = \lfloor 11.0 \rfloor = \mathbf{11}\text{ whole models}$$

**Answer format:** `GB_f32, GB_f16, GB_u8, GB_A_naive, GB_B_naive, fits, accumulator_GB, passes_A, passes_B, min_A, min_B, max_models_3view, max_models_2view`
**Final Result:** `8.192, 4.096, 2.048, 73.728, 32.768, neither, 8.192, 45000, 20000, 9.0, 4.0, 7, 11`
*(Rounded variant: `8.2, 4.10, 2.05, 73.7, 32.8, neither, 8.2, 45000, 20000, 9.0, 4.0, 7, 11`)*

In [10]:
# ==============================================================================
# Question 8: Inference Budget: Memory and Time Across Two Ensemble Configurations
# ==============================================================================
n_images = 5000
h, w = 128, 800
n_classes = 4
t_per_pass_s = 0.012  # 12 ms

# 1. Memory for 1 set of test-set probability maps (decimal units: 1 GB = 1e9 bytes)
total_elements = n_images * n_classes * h * w  # 2,048,000,000
GB_f32 = (total_elements * 4) / 1e9
GB_f16 = (total_elements * 2) / 1e9
GB_u8  = (total_elements * 1) / 1e9

# 2. Naive memory consumption vs. 16 GB budget
GB_A_naive = 3 * 3 * GB_f32
GB_B_naive = 2 * 2 * GB_f32
fits = "neither"  # Both 73.7 GB and 32.8 GB exceed 16 GB
accumulator_GB = GB_f32

# 3. Forward passes and model execution time
passes_A = n_images * 3 * 3
passes_B = n_images * 2 * 2
min_A = (passes_A * t_per_pass_s) / 60.0
min_B = (passes_B * t_per_pass_s) / 60.0

# 4. Maximum whole models under 30 min budget (8 min I/O -> 22 min compute)
avail_s = (30 - 8) * 60.0  # 1320 s
time_per_model_3view_s = (n_images * 3) * t_per_pass_s  # 180 s (3.0 min)
time_per_model_2view_s = (n_images * 2) * t_per_pass_s  # 120 s (2.0 min)
max_models_3view = int(avail_s // time_per_model_3view_s)
max_models_2view = int(avail_s // time_per_model_2view_s)

print("--- Memory Requirements (Decimal Units: 1 GB = 10^9 bytes) ---")
print(f"1 Full Set: float32 = {GB_f32:.3f} GB | float16 = {GB_f16:.3f} GB | uint8 = {GB_u8:.3f} GB")
print(f"Config A Naive: {GB_A_naive:.3f} GB | Config B Naive: {GB_B_naive:.3f} GB | Fits 16GB: {fits}")
print(f"Running Accumulator: {accumulator_GB:.3f} GB")

print("\n--- Latency & Passes ---")
print(f"Config A: {passes_A:,} passes, {min_A:.1f} minutes")
print(f"Config B: {passes_B:,} passes, {min_B:.1f} minutes")

print("\n--- Model Ensembling Limits (22 min compute budget) ---")
print(f"Max Whole Models (3-view TTA): {max_models_3view}")
print(f"Max Whole Models (2-view TTA): {max_models_2view}")

# Formatted Answers
q8_answer_exact = f"{GB_f32:.3f}, {GB_f16:.3f}, {GB_u8:.3f}, {GB_A_naive:.3f}, {GB_B_naive:.3f}, {fits}, {accumulator_GB:.3f}, {passes_A}, {passes_B}, {min_A:.1f}, {min_B:.1f}, {max_models_3view}, {max_models_2view}"
q8_answer_rounded = f"{GB_f32:.1f}, {GB_f16:.2f}, {GB_u8:.2f}, {GB_A_naive:.1f}, {GB_B_naive:.1f}, {fits}, {accumulator_GB:.1f}, {passes_A}, {passes_B}, {min_A:.1f}, {min_B:.1f}, {max_models_3view}, {max_models_2view}"
print("\n" + "="*60)
print(f"Question 8 Final Answer (exact):   {q8_answer_exact}")
print(f"Question 8 Final Answer (rounded): {q8_answer_rounded}")
print("="*60)


--- Memory Requirements (Decimal Units: 1 GB = 10^9 bytes) ---
1 Full Set: float32 = 8.192 GB | float16 = 4.096 GB | uint8 = 2.048 GB
Config A Naive: 73.728 GB | Config B Naive: 32.768 GB | Fits 16GB: neither
Running Accumulator: 8.192 GB

--- Latency & Passes ---
Config A: 45,000 passes, 9.0 minutes
Config B: 20,000 passes, 4.0 minutes

--- Model Ensembling Limits (22 min compute budget) ---
Max Whole Models (3-view TTA): 7
Max Whole Models (2-view TTA): 11

Question 8 Final Answer (exact):   8.192, 4.096, 2.048, 73.728, 32.768, neither, 8.192, 45000, 20000, 9.0, 4.0, 7, 11
Question 8 Final Answer (rounded): 8.2, 4.10, 2.05, 73.7, 32.8, neither, 8.2, 45000, 20000, 9.0, 4.0, 7, 11


---
## Question 9: Designing the Final Inference Notebook

### Problem Statement
Comprehensive architectural design of the final competition inference notebook: pipeline step ordering, avoidance of threshold/area overfitting, empty mask representation, and pre-submission safety checklist.

### 1. Correct Order of Pipeline Steps & Swapping Failure Analysis
- **Canonical Pipeline Sequence**:
  1. **Raw Image Loading & Preprocessing**: Load image, resize/pad to input canvas ($128 \times 800$), normalize with ImageNet statistics.
  2. **Test-Time Augmentation (TTA) Generation**: Create augmented tensor variations (e.g., horizontal flip on `dims=[3]`, vertical flip on `dims=[2]`).
  3. **Multi-Model Forward Passes**: Feed each view through each ensemble model under `torch.inference_mode()` and mixed precision (`torch.autocast`).
  4. **Spatial Inverse Transformation (Un-flipping)**: Apply exact inverse spatial operations (`torch.flip`) to restore augmented probability tensors into original coordinate alignment.
  5. **Soft Ensemble Probability Averaging**: Compute weighted or arithmetic mean across models and views in continuous probability space: $\bar{\mathbf{P}} = \frac{1}{K} \sum \mathbf{P}_k$.
  6. **Per-Class Calibrated Thresholding**: Binarize continuous probabilities into discrete candidate masks using class-specific cutoffs: $\hat{\mathbf{X}}_c = (\bar{\mathbf{P}}_c > \tau_c)$.
  7. **Connected-Component Minimum-Area Filtering**: Run 8-connected component analysis (`cv2.connectedComponentsWithStats`) and discard blobs with area $< A_{\min, c}$.
  8. **Fortran-Ordered Run-Length Encoding (RLE) & CSV Serialization**: Flatten masks using `order='F'`, clamp indices $\le 102,400$, encode 1-based start-lengths, format empty masks with sentinel, and stream to `submission.csv`.

- **Why Swapping Any Two Adjacent Steps is a Fatal Bug**:
  - *Swapping Step 3 and Step 4 (Un-flipping after Averaging - The "Forgot to Un-flip" Bug)*: Averaging raw flipped views directly blurs spatially inverted features across the sheet (e.g. left edge blurs with right edge), destroying localization and collapsing peak confidence.
  - *Swapping Step 5 and Step 6 (Thresholding before Averaging - Hard Voting)*: Converts continuous probabilities into rigid binary masks before blending. Hard voting discards confidence margins, causes ties, and severely underperforms soft probability averaging.
  - *Swapping Step 6 and Step 7 (Area Filtering before Thresholding)*: Connected-component analysis requires discrete binary $\{0, 1\}$ inputs; executing it on continuous probabilities is mathematically invalid.
  - *Swapping Step 7 and Step 8 (RLE Encoding before Area Filtering)*: Post-processing RLE strings directly cannot perform morphological neighborhood analysis without expensive, repeated decompression loops.

### 2. Preventing Overfitting & Connection to the All-Empty Baseline
- **Two Ways to Avoid Overfitting Post-Processing Hyperparameters**:
  1. *Out-of-Fold Cross-Validation / Stratified Held-out Verification*: Optimize thresholds $\boldsymbol{\tau}$ and area filters $\mathbf{A}_{\min}$ strictly using out-of-fold validation splits across multiple folds, rather than fitting a single noisy validation set.
  2. *Coarse Search Grids & Physical Morphology Constraints*: Restrict search grids to coarse intervals (e.g., $\Delta \tau = 0.05$) to prevent fitting sample noise, and set area filter ceilings strictly below known minimum true defect sizes (e.g. preserving fine scratches and rare `defect_b`).
- **Connection to the All-Empty Baseline (Milestone 1, Question 9)**:
  - In industrial defect detection, ~84% of test image-class pairs are completely defect-free.
  - The all-empty baseline earns a high Empty Accuracy ($1.0$) and False Positive Score ($1.0$), scoring $0.3000$ with zero defect detection capability.
  - Overtuning high thresholds or aggressive area filters deceptively boosts the validation score by wiping borderline predictions into empty masks; however, this destroys true positive recall on rare defects and causes catastrophic collapse on the private test leaderboard.

### 3. Submission Integrity & Pre-Submission Safety Checklist
- **Empty Mask Representation in Submission**:
  - Completely empty masks must be serialized with the exact single whitespace sentinel string: `" "` (or empty string with strict quote handling, preventing Pandas `,,` null-field crashes).
- **Safety Checklist Before Pressing 'Submit'**:
  1. *Memory & Streaming Architecture Audit*: Verify test inference processes minibatches in bounded $O(1)$ RAM ($B=16$), avoids monolithic full-test tensor allocations, and explicitly invokes `del` and `gc.collect()` to stay comfortably within the 16 GB RAM / 20 GB disk envelope.
  2. *Automated 7-Point Schema & Row Count Validation*: Confirm `submission.csv` contains exactly 12,001 rows (1 header + 12,000 predictions for 3,000 images $\times$ 4 classes), sequential 1-based IDs, valid defect class cycles, non-null entries, and run starts $\le 102,400$.
  3. *Latency Budget Extrapolation*: Profile inference on the first 100 test images to ensure the full test set will execute in $< 20$ minutes, well within Kaggle's notebook timeout ceiling.